In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if not (project_root / "app.py").exists() and (project_root.parent / "app.py").exists():
    project_root = project_root.parent
    print('project_root: ', project_root)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

project_root:  C:\Program Files\Studying\coding\RAG_project


# Inference Evaluation Notebook
This notebook provides an **industry-style evaluation workflow** for inference runs created from either:
- `notebook/inference.ipynb`
- Chainlit UI in `app.py`

## What You Can Get Reliably After Inference
From saved session JSON files, you can reliably retrieve:
- session/chat id
- LLM provider and LLM model
- full turn history (user and assistant messages)
- timestamp and environment metadata

## What Is Not Persisted by Default
Retriever top-k results are not fully persisted in session JSON by default. For that, this notebook offers:
- **Replay analysis**: rerun retrieval on each user turn to inspect retrieved nodes
- **Optional live capture**: instrument a temporary query path to capture retrieval context during evaluation

Use this notebook when you want structured analysis tables and turn-by-turn diagnostics.

In [3]:
from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import Any
import re

import pandas as pd

from src.rag.inference import (
    DEFAULT_COLLECTION_NAME,
    DEFAULT_EMBED_MODEL_NAME,
    DEFAULT_HUGGINGFACE_MODEL_KEY,
    DEFAULT_LLM_PROVIDER,
    DEFAULT_OLLAMA_MODEL,
    create_rag_app,
    default_paths,
    HUGGINGFACE_CHAT_MODELS,
)


PROJECT_ROOT = project_root
paths = default_paths(PROJECT_ROOT)
SESSION_DIR = paths.session_dir
CHROMA_DIR = paths.chroma_dir

COLLECTION_NAME = DEFAULT_COLLECTION_NAME
EMBED_MODEL_NAME = DEFAULT_EMBED_MODEL_NAME
LLM_PROVIDER = DEFAULT_LLM_PROVIDER
OLLAMA_MODEL = DEFAULT_OLLAMA_MODEL
HUGGINGFACE_MODEL = DEFAULT_HUGGINGFACE_MODEL_KEY
HUGGINGFACE_PROVIDER = "auto"


@dataclass(frozen=True)
class EvaluationConfig:
    """Configuration object for evaluation.

    Purpose:
    - Keep all runtime knobs in one typed object to reduce hidden global state.

    Output:
    - Immutable configuration record used by helper functions.
    """

    project_root: Path
    session_dir: Path
    chroma_dir: Path
    collection_name: str
    embed_model_name: str
    llm_provider: str
    ollama_model: str
    huggingface_model: str
    huggingface_provider: str


EVAL_CONFIG = EvaluationConfig(
    project_root=PROJECT_ROOT,
    session_dir=SESSION_DIR,
    chroma_dir=CHROMA_DIR,
    collection_name=COLLECTION_NAME,
    embed_model_name=EMBED_MODEL_NAME,
    llm_provider=LLM_PROVIDER,
    ollama_model=OLLAMA_MODEL,
    huggingface_model=HUGGINGFACE_MODEL,
    huggingface_provider=HUGGINGFACE_PROVIDER,
)


def _safe_read_json(path: Path) -> dict[str, Any] | None:
    """Load a JSON file safely.

    Purpose:
    - Avoid notebook crashes when one session file is malformed.

    Output:
    - Parsed dictionary if valid.
    - None if the file cannot be parsed.
    """
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None


def list_saved_session_files(session_dir: Path) -> list[Path]:
    """List saved inference sessions in newest-first order.

    Purpose:
    - Provide a deterministic file ordering for dashboard and reporting.

    Output:
    - List of session JSON file paths sorted by modification time descending.
    """
    return sorted(session_dir.glob("*.json"), key=lambda p: p.stat().st_mtime, reverse=True)


def build_session_catalog(session_dir: Path) -> pd.DataFrame:
    """Create a table-level overview of all saved chat sessions.

    Purpose:
    - Build one row per session file so you can filter by chat id, model, or date.

    Output:
    - pandas DataFrame with metadata columns (chat_id, model, message_count, etc.).
    """
    rows: list[dict[str, Any]] = []
    for session_path in list_saved_session_files(session_dir):
        payload = _safe_read_json(session_path)
        if payload is None:
            rows.append(
                {
                    "file_name": session_path.name,
                    "chat_id": None,
                    "status": "invalid_json",
                    "message_count": None,
                    "llm_provider": None,
                    "llm_model": None,
                    "updated_at": None,
                }
            )
            continue

        messages = payload.get("messages", []) or []
        rows.append(
            {
                "file_name": session_path.name,
                "chat_id": payload.get("chat_id", session_path.stem),
                "status": "ok",
                "message_count": len(messages),
                "llm_provider": payload.get("llm_provider"),
                "llm_model": payload.get("llm_model"),
                "embedding_model": payload.get("embedding_model"),
                "collection_name": payload.get("collection_name"),
                "updated_at": payload.get("updated_at"),
            }
        )

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(by=["updated_at", "file_name"], ascending=[False, True], na_position="last")
    return df.reset_index(drop=True)


def load_chat_payload(chat_id: str, session_dir: Path) -> dict[str, Any]:
    """Load one chat payload by chat_id.

    Purpose:
    - Resolve the expected file path and return the persisted session payload.

    Output:
    - Dictionary containing full session metadata and message history.
    - Raises FileNotFoundError if the chat file does not exist.
    """
    safe_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", chat_id.strip()).strip("_") or "default_chat"
    session_path = session_dir / f"{safe_name}.json"
    if not session_path.exists():
        raise FileNotFoundError(f"Session file not found for chat_id={chat_id}: {session_path}")

    payload = _safe_read_json(session_path)
    if payload is None:
        raise ValueError(f"Session file is not valid JSON: {session_path}")
    return payload


def build_turn_table(payload: dict[str, Any]) -> pd.DataFrame:
    """Transform one session payload into turn-level tabular data.

    Purpose:
    - Convert raw message array into analysis-friendly rows with turn index and role.

    Output:
    - DataFrame with one row per message and extracted text statistics.
    """
    rows: list[dict[str, Any]] = []
    messages = payload.get("messages", []) or []
    for idx, item in enumerate(messages, start=1):
        content = (item.get("content") or "").strip()
        rows.append(
            {
                "chat_id": payload.get("chat_id"),
                "turn_index": idx,
                "role": item.get("role"),
                "content": content,
                "content_length": len(content),
                "updated_at": payload.get("updated_at"),
            }
        )
    return pd.DataFrame(rows)


def build_user_assistant_pairs(turn_df: pd.DataFrame) -> pd.DataFrame:
    """Pair each user message with the following assistant response.

    Purpose:
    - Create a clean unit-of-analysis for quality review and error analysis.

    Output:
    - DataFrame with pair_id, user_message, assistant_message, and text lengths.
    """
    if turn_df.empty:
        return pd.DataFrame(
            columns=[
                "pair_id",
                "chat_id",
                "user_turn",
                "assistant_turn",
                "user_message",
                "assistant_message",
                "user_length",
                "assistant_length",
            ]
        )

    rows: list[dict[str, Any]] = []
    pair_id = 0
    user_buffer: dict[str, Any] | None = None

    for row in turn_df.to_dict(orient="records"):
        role = (row.get("role") or "").lower()
        if role == "user":
            user_buffer = row
            continue
        if role == "assistant" and user_buffer is not None:
            pair_id += 1
            rows.append(
                {
                    "pair_id": pair_id,
                    "chat_id": row.get("chat_id"),
                    "user_turn": user_buffer.get("turn_index"),
                    "assistant_turn": row.get("turn_index"),
                    "user_message": user_buffer.get("content"),
                    "assistant_message": row.get("content"),
                    "user_length": user_buffer.get("content_length"),
                    "assistant_length": row.get("content_length"),
                }
            )
            user_buffer = None

    return pd.DataFrame(rows)


catalog_df = build_session_catalog(EVAL_CONFIG.session_dir)
print(f"Session directory: {EVAL_CONFIG.session_dir}")
print(f"Detected session files: {len(catalog_df)}")
catalog_df.head(20)

resource module not available on Windows
Session directory: C:\Program Files\Studying\coding\RAG_project\session
Detected session files: 2


,file_name,chat_id,status,message_count,llm_provider,llm_model,embedding_model,collection_name,updated_at
0,Test_Chat_session.json,Test_Chat_session,ok,14,ollama,gemma3:1b,BAAI/bge-small-en-v1.5,run_testing,2026-07-10T05:29:56.126527+00:00
1,Donald_Trump.json,Donald_Trump,ok,26,ollama,gemma3:1b,BAAI/bge-small-en-v1.5,run_testing,2026-07-06T06:46:18.529188+00:00


In [ ]:
# --- Example usage: pick one chat id from catalog_df, then run this cell ---

# Replace this with an id from catalog_df.chat_id
# TARGET_CHAT_ID = (catalog_df["chat_id"].dropna().iloc[0] if not catalog_df.empty else None)
TARGET_CHAT_ID = Test_chat_session

def resolve_llm_label(llm_provider: str, llm_model: str) -> str:
    """Create a display label for model reporting.

    Purpose:
    - Normalize model naming for consistent dashboards and exports.

    Output:
    - Human-friendly model label string.
    """
    if llm_provider == "huggingface":
        reverse_lookup = {v: k for k, v in HUGGINGFACE_CHAT_MODELS.items()}
        model_key = reverse_lookup.get(llm_model, llm_model)
        return f"huggingface::{model_key}"
    return f"{llm_provider}::{llm_model}"


def replay_retrieval_for_chat(
    chat_id: str,
    payload: dict[str, Any],
    config: EvaluationConfig,
    top_k: int = 5,
    include_assistant_regeneration: bool = False,
) -> pd.DataFrame:
    """Replay retrieval for user turns to inspect top retrieved nodes.

    Purpose:
    - Retrieve per-turn context candidates for analysis when original retriever results
      were not persisted in session JSON.
    - Optionally regenerate assistant answers for side-by-side comparison.

    Output:
    - DataFrame with one row per retrieved node per user turn.
    """
    rag_app = create_rag_app(
        chroma_dir=config.chroma_dir,
        session_dir=config.session_dir,
        collection_name=config.collection_name,
        embed_model_name=config.embed_model_name,
        llm_provider=config.llm_provider,
        ollama_model=config.ollama_model,
        huggingface_model=config.huggingface_model,
        huggingface_provider=config.huggingface_provider,
        final_top_k=top_k,
    )

    # Open an isolated chat id to avoid modifying production session state during replay.
    replay_chat_id = f"{chat_id}__evaluation_replay"
    rag_app.open_chat(replay_chat_id, load_existing=False, overwrite=True)

    rows: list[dict[str, Any]] = []
    messages = payload.get("messages", []) or []
    user_turn_number = 0

    for item in messages:
        role = (item.get("role") or "").lower()
        content = (item.get("content") or "").strip()
        if not content:
            continue

        if role == "user":
            user_turn_number += 1
            retrieved = rag_app.hybrid_retriever.retrieve(content)
            if include_assistant_regeneration:
                regenerated = rag_app.chat(replay_chat_id, content)
                regenerated_text = getattr(regenerated, "response", "") or ""
            else:
                regenerated_text = ""

            for rank, node_with_score in enumerate(retrieved, start=1):
                node = node_with_score.node
                node_text = (node.get_content() or "").strip()
                metadata = node.metadata or {}
                rows.append(
                    {
                        "chat_id": chat_id,
                        "user_turn": user_turn_number,
                        "rank": rank,
                        "retrieval_score": node_with_score.score,
                        "node_id": node.node_id,
                        "article_date": metadata.get("article_date"),
                        "article_title": metadata.get("article_title", metadata.get("file_name")),
                        "node_text_preview": node_text[:300],
                        "regenerated_assistant_text": regenerated_text,
                    }
                )

    return pd.DataFrame(rows)


if TARGET_CHAT_ID is None:
    print("No saved chats found yet. Run inference first in inference.ipynb or app.py.")
else:
    payload = load_chat_payload(TARGET_CHAT_ID, EVAL_CONFIG.session_dir)
    turn_df = build_turn_table(payload)
    pair_df = build_user_assistant_pairs(turn_df)

    print("=== Session Metadata ===")
    print(f"chat_id:        {payload.get('chat_id')}")
    print(f"collection:     {payload.get('collection_name')}")
    print(f"embedding:      {payload.get('embedding_model')}")
    print(f"llm_provider:   {payload.get('llm_provider')}")
    print(f"llm_model:      {payload.get('llm_model')}")
    print(f"llm_label:      {resolve_llm_label(payload.get('llm_provider'), payload.get('llm_model'))}")
    print(f"updated_at:     {payload.get('updated_at')}")
    print(f"message_count:  {len(payload.get('messages', []))}")

    print("\n=== Turn Table (first rows) ===")
    display(turn_df.head(20))

    print("\n=== User/Assistant Pairs (first rows) ===")
    display(pair_df.head(20))

    print("\n=== Replay Retrieval Diagnostics (top nodes) ===")
    retrieval_df = replay_retrieval_for_chat(
        chat_id=TARGET_CHAT_ID,
        payload=payload,
        config=EVAL_CONFIG,
        top_k=5,
        include_assistant_regeneration=False,
    )
    display(retrieval_df.head(30))

    # Optional: persist evaluation artifacts for audit trail.
    output_dir = PROJECT_ROOT / "documents" / "evaluation_outputs"
    output_dir.mkdir(parents=True, exist_ok=True)
    turn_df.to_csv(output_dir / f"{TARGET_CHAT_ID}_turns.csv", index=False, encoding="utf-8")
    pair_df.to_csv(output_dir / f"{TARGET_CHAT_ID}_pairs.csv", index=False, encoding="utf-8")
    retrieval_df.to_csv(output_dir / f"{TARGET_CHAT_ID}_retrieval_replay.csv", index=False, encoding="utf-8")
    print(f"Saved evaluation CSV files to: {output_dir}")